# Modelo de Predicción de ROI — Procesos RPA (XGBoost + GPU)

**Objetivo:** Entrenar un modelo de machine learning que prediga el ROI (%) de una automatización RPA antes de implementarla, a partir de características observables del proceso.

**Algoritmo:** XGBoostRegressor con aceleración GPU (CUDA). Detecta automáticamente RTX 4060 y hace fallback a CPU si no hay GPU disponible.

**Flujo del notebook:**
1. Carga del dataset de ROI calculado
2. Análisis de features y target
3. Comparación de algoritmos (Ridge, RF, GBM, XGBoost CPU vs GPU)
4. Entrenamiento final con XGBoost + GPU
5. Importancia de variables
6. Guardado del modelo para producción

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.utils.roi_calculator import build_roi_dataset
from src.models.roi_predictor import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET,
    prepare_features, train, get_feature_importance, _detect_device,
)

pd.set_option('display.float_format', '{:,.3f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Detectar GPU
DEVICE = _detect_device()
print(f'XGBoost version: {xgb.__version__}')
print(f'Dispositivo disponible: {DEVICE.upper()}')
if DEVICE == 'cuda':
    print('  RTX 4060 detectada — entrenamiento en GPU activado.')

## 1. Carga y preparación del dataset

In [ ]:
df_raw = build_roi_dataset()
df_model = df_raw.dropna(subset=[TARGET, 'TiempoManualHoras', 'ValorHoraPromedio']).copy()

print(f'Filas totales:              {len(df_raw)}')
print(f'Filas con ROI calculable:   {len(df_model)}')
print(f'Target — ROI_Porcentaje:')
print(df_model[TARGET].describe().to_string())

In [ ]:
# Distribución del target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_model[TARGET], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución ROI (%) — target')
axes[0].set_xlabel('ROI (%)')

# Log-transform
roi_pos = df_model[df_model[TARGET] > 0][TARGET]
axes[1].hist(np.log1p(roi_pos), bins=15, color='seagreen', edgecolor='white')
axes[1].set_title('Distribución log(1 + ROI) — bots con ROI > 0')
axes[1].set_xlabel('log(1 + ROI)')

plt.tight_layout()
plt.show()

In [ ]:
X = prepare_features(df_model)
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras')
print(f'Features numéricas:    {len(NUMERIC_FEATURES)}')
print(f'Features categóricas:  {len(CATEGORICAL_FEATURES)}')
print(f'\nValores nulos en X_train: {X_train.isnull().sum().sum()}')

## 2. Comparación de algoritmos

In [ ]:
def make_pipeline(estimator):
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([('pre', preprocessor), ('model', estimator)])

# Usar log1p(ROI) para comparar todos los algoritmos en la misma escala
df_pos = df_model[df_model[TARGET] > 0].copy()
X_cmp = prepare_features(df_pos)
y_cmp = np.log1p(df_pos[TARGET])

candidates = {
    'Ridge':            make_pipeline(Ridge(alpha=10)),
    'RandomForest':     make_pipeline(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    'GradientBoosting': make_pipeline(GradientBoostingRegressor(
                            n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
    'XGBoost (CPU)':    make_pipeline(xgb.XGBRegressor(
                            n_estimators=300, learning_rate=0.05, max_depth=3,
                            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                            random_state=42, device='cpu', tree_method='hist', verbosity=0)),
    'XGBoost (GPU)':    make_pipeline(xgb.XGBRegressor(
                            n_estimators=300, learning_rate=0.05, max_depth=3,
                            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                            random_state=42, device=DEVICE, tree_method='hist', verbosity=0)),
}

n_cv = min(5, max(2, len(X_cmp) // 8))
kf = KFold(n_splits=n_cv, shuffle=True, random_state=42)
results = []

for name, pipe in candidates.items():
    t0 = time.time()
    cv_r2 = cross_val_score(pipe, X_cmp, y_cmp, cv=kf, scoring='r2')
    elapsed = time.time() - t0
    results.append({
        'Modelo': name,
        'CV R² (media)': cv_r2.mean(),
        'CV R² (std)': cv_r2.std(),
        'Tiempo (s)': round(elapsed, 2),
    })
    print(f'{name:<22} R²={cv_r2.mean():.3f} ± {cv_r2.std():.3f}  |  {elapsed:.2f}s')

df_results = pd.DataFrame(results).sort_values('CV R² (media)', ascending=False)
df_results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_results))]
ax.barh(df_results['Modelo'], df_results['CV R² (media)'], xerr=df_results['CV R² (std)'],
        color=colors, capsize=4, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Comparación de modelos — R² (CV 5-fold)')
ax.set_xlabel('R² promedio')
plt.tight_layout()
plt.show()

best_model_name = df_results.iloc[0]['Modelo']
print(f'\nMejor modelo: {best_model_name}')

## 3. Entrenamiento final con XGBoost + GPU

In [ ]:
t0 = time.time()
metrics = train(df_model)
elapsed = time.time() - t0

print(f'=== Entrenamiento XGBoost ({metrics["device"].upper()}) ===')
print(f'  Dispositivo:         {metrics["device"].upper()}')
print(f'  Tiempo total:        {elapsed:.2f}s')
print(f'  R² (test, log):      {metrics["r2"]:.4f}')
print(f'  MAE (test, orig):    {metrics["mae_pct"]:,.0f}%')
print(f'  RMSE(test, orig):    {metrics["rmse_pct"]:,.0f}%')
print(f'  R² CV ({min(5, max(2, len(df_model)//8))}-fold):    {metrics["cv_r2_mean"]:.4f} ± {metrics["cv_r2_std"]:.4f}')
print(f'  Muestras train/test: {metrics["n_train"]} / {metrics["n_test"]}')

In [ ]:
from src.models.roi_predictor import load_model

artifact = load_model()
pipeline = artifact["pipeline"]

df_pos2 = df_model[df_model[TARGET] > 0].copy()
X_test_all = prepare_features(df_pos2)
y_log_all  = np.log1p(df_pos2[TARGET])
y_pred_log = pipeline.predict(X_test_all)
y_pred_raw = np.expm1(y_pred_log)
y_raw      = df_pos2[TARGET].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_raw, y_pred_raw, alpha=0.6, color='#db0061', edgecolors='white', s=60)
lim = [min(y_raw.min(), y_pred_raw.min()), max(y_raw.max(), y_pred_raw.max())]
axes[0].plot(lim, lim, 'k--', label='Predicción perfecta')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('ROI real (%) — escala log')
axes[0].set_ylabel('ROI predicho (%) — escala log')
axes[0].set_title(f'Predicho vs Real (R²={metrics["r2"]:.3f})')
axes[0].legend()

residuos_log = y_pred_log - y_log_all.values
axes[1].hist(residuos_log, bins=15, color='#3c3f52', edgecolor='white')
axes[1].axvline(0, color='#db0061', linestyle='--')
axes[1].set_title('Residuos en escala log(ROI)')
axes[1].set_xlabel('log(predicho) − log(real)')

plt.tight_layout()
plt.savefig('../reports/figures/modelo_evaluacion.png', bbox_inches='tight')
plt.show()

## 4. Importancia de variables

In [ ]:
df_imp = get_feature_importance(top_n=15)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#db0061' if i < 5 else '#3c3f52' for i in range(len(df_imp))]
ax.barh(df_imp['feature'][::-1], df_imp['importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title(f'Importancia de variables — XGBoost ({metrics["device"].upper()})')
ax.set_xlabel('Importancia relativa (gain)')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 factores que más explican el ROI:')
for _, row in df_imp.head(5).iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

## 5. Predicciones para todos los bots del portafolio

In [ ]:
X_all = prepare_features(df_model)
df_model['ROI_Predicho'] = pipeline.predict(X_all)
df_model['Error_Prediccion'] = df_model['ROI_Predicho'] - df_model['ROI_Porcentaje']

comparacion = df_model[['Automatizacion','ROI_Porcentaje','ROI_Predicho','Error_Prediccion']].copy()
comparacion = comparacion.sort_values('ROI_Porcentaje', ascending=False)
comparacion['ROI_Porcentaje'] = comparacion['ROI_Porcentaje'].round(1)
comparacion['ROI_Predicho'] = comparacion['ROI_Predicho'].round(1)
comparacion['Error_Prediccion'] = comparacion['Error_Prediccion'].round(1)

print('Predicción vs. ROI real (todos los bots):')
comparacion

## 6. Ejemplo de predicción para un nuevo bot

In [ ]:
from src.models.roi_predictor import predict

nuevo_bot = {
    'TiempoManualHoras':     3.0,     # 3 horas de trabajo manual por ejecución
    'Num_Ejecuciones':       200,     # 200 ejecuciones esperadas
    'ValorHoraPromedio':     35000,   # $35,000 COP/hora
    'Tecnologia':            'UiPath',
    'Estado':                'Activo',
    'DuracionPromedio_Horas': 0.25,   # Robot tarda 15 minutos
    'PromTransacciones':     10,
    'TasaExito':             0.95,
    'TasaError':             0.02,
    'EjecucionesPorDia':     200/365,
    'DiasEnProduccion':      365,
    'NumAreas':              2,
    'NumRoles':              1,
}

resultado = predict(nuevo_bot)

print('=== Predicción para nuevo bot ===')
print(f"  ROI predicho:         {resultado['roi_porcentaje']:.0f}%")
print(f"  Ahorro neto:          ${resultado['ahorro_neto_cop']/1e6:.2f}M COP")
print(f"  Beneficio bruto:      ${resultado['beneficio_bruto_cop']/1e6:.2f}M COP")
print(f"  Costo robot:          ${resultado['costo_robot_cop']/1e6:.2f}M COP")

if resultado['roi_porcentaje'] > 500:
    print('\nVeredicto: EXCELENTE candidato para automatización.')
elif resultado['roi_porcentaje'] > 100:
    print('\nVeredicto: BUENA candidata para automatización.')
else:
    print('\nVeredicto: Evaluar con cuidado antes de automatizar.')

In [ ]:
print(f'Modelo XGBoost ({metrics["device"].upper()}) guardado en: models/roi_model.joblib')
print('Listo para usar desde la aplicación Streamlit (app/chat.py)')